# Notebook 17 — GRPO Foundations and Reward Design

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/finetuning/17_grpo_foundations_and_reward_design.ipynb)

GRPO is an advanced post-training tool for cases where simple labels are not enough but reliable reward signals still exist. This notebook stays deliberately small and transparent: local toy tasks, inspectable verifiers, explicit reward functions, and concrete reasons RL is powerful **and** why it should not be your first lever.

## What you will build

- a plain-language map of SFT, RLHF, RLVR, and GRPO
- a tiny verifier-friendly task suite
- explicit reward components you can inspect and change
- group-relative scoring that explains the *group* in GRPO
- a checklist for when RL is justified and when it is not

## Position in the training stack

In this course, RL comes after:

1. prompt improvements
2. better retrieval or tools
3. cleaner supervised data
4. stronger evaluation
5. preference alignment where it fits

Only then does GRPO become attractive for verifier-rich tasks such as arithmetic, code execution, schema adherence, and tool correctness.

In [ ]:
# --- Google Colab Pro Fine-Tuning Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## Step 1: Add notebook helpers and artifact paths

We will keep the notebook runnable with standard-library helpers, readable tables, and small artifacts saved locally.

In [ ]:
from collections import Counter
import statistics
import re

random.seed(17)

ARTIFACT_DIR = Path("artifacts") / "notebook_17_grpo_foundations"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def to_markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    return "\n".join([header, divider, *body])

def ascii_bar(value, width=16, filled="█"):
    value = max(0.0, min(1.0, value))
    used = int(round(value * width))
    return filled * used + "·" * (width - used)

def softmax_dict(weights):
    largest = max(weights.values())
    exps = {key: math.exp(value - largest) for key, value in weights.items()}
    total = sum(exps.values())
    return {key: value / total for key, value in exps.items()}

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Compare the main post-training options

The key question is not "is RL powerful?" It is "what supervision signal do I actually have?"

In [ ]:
post_training_options = [
    {
        "method": "SFT",
        "signal": "gold targets",
        "best_for": "clear demonstrations",
        "strength": "cheap and stable",
        "risk": "copies label mistakes",
    },
    {
        "method": "DPO / ORPO / KTO",
        "signal": "preferences",
        "best_for": "ranking better vs worse responses",
        "strength": "simple second stage after SFT",
        "risk": "depends on preference quality",
    },
    {
        "method": "RLHF",
        "signal": "learned reward model",
        "best_for": "subjective quality with many comparisons",
        "strength": "captures soft preferences",
        "risk": "reward-model drift and complexity",
    },
    {
        "method": "RLVR",
        "signal": "programmatic verifier",
        "best_for": "verifiable tasks like math or schema checks",
        "strength": "cheap, auditable rewards",
        "risk": "narrow reward coverage",
    },
    {
        "method": "GRPO",
        "signal": "group-relative rewards",
        "best_for": "reasoning tasks with multiple sampled rollouts",
        "strength": "stabilizes updates without a separate critic",
        "risk": "reward hacking and rollout cost",
    },
]

print(to_markdown_table(post_training_options, ["method", "signal", "best_for", "strength", "risk"]))

## Step 3: Score when RL is justified

RL makes sense when you have:

- a target behavior that static labels miss
- a reward signal you trust
- tasks that can be checked cheaply
- strong evaluation to catch regressions

The next cell turns that into an explicit checklist.

In [ ]:
decision_cases = [
    {
        "case": "Support style rewrite",
        "verifiable": 0.1,
        "baseline_gap": 0.3,
        "reward_quality": 0.2,
        "eval_maturity": 0.6,
        "notes": "Mostly a data and rubric problem",
    },
    {
        "case": "Math tutor with exact answers",
        "verifiable": 1.0,
        "baseline_gap": 0.8,
        "reward_quality": 0.9,
        "eval_maturity": 0.8,
        "notes": "Strong candidate for GRPO",
    },
    {
        "case": "JSON tool calls with schemas",
        "verifiable": 0.9,
        "baseline_gap": 0.7,
        "reward_quality": 0.85,
        "eval_maturity": 0.75,
        "notes": "Good RLVR / GRPO territory",
    },
    {
        "case": "General friendliness tuning",
        "verifiable": 0.2,
        "baseline_gap": 0.4,
        "reward_quality": 0.25,
        "eval_maturity": 0.5,
        "notes": "Prefer SFT + preferences first",
    },
]

def rl_readiness_score(case):
    score = (
        0.35 * case["verifiable"]
        + 0.25 * case["reward_quality"]
        + 0.20 * case["baseline_gap"]
        + 0.20 * case["eval_maturity"]
    )
    return round(score, 3)

for case in decision_cases:
    case["rl_score"] = rl_readiness_score(case)
    case["recommendation"] = "use RL later" if case["rl_score"] < 0.6 else "RL is plausible"
    case["score_bar"] = ascii_bar(case["rl_score"])

print(to_markdown_table(decision_cases, ["case", "rl_score", "score_bar", "recommendation", "notes"]))

## Step 4: Create a tiny verifier-friendly task set

These toy tasks are intentionally local and inspectable. They are the kind of exercises that make RL tractable because correctness can be checked with code.

In [ ]:
toy_tasks = [
    {
        "id": "sum_12_19",
        "category": "arithmetic",
        "prompt": "Compute 12 + 19. Respond with reasoning and a final line exactly as FINAL: <number>.",
        "answer": "31",
        "verifier": "exact_integer",
    },
    {
        "id": "parity_41",
        "category": "parity",
        "prompt": "Is 41 odd or even? End with FINAL: odd or FINAL: even.",
        "answer": "odd",
        "verifier": "exact_label",
    },
    {
        "id": "sort_3_1_2",
        "category": "ordering",
        "prompt": "Sort the numbers 3, 1, 2 ascending and finish with FINAL: 1,2,3.",
        "answer": "1,2,3",
        "verifier": "exact_sequence",
    },
    {
        "id": "mul_7_8",
        "category": "arithmetic",
        "prompt": "Compute 7 * 8 and end with FINAL: <number>.",
        "answer": "56",
        "verifier": "exact_integer",
    },
]

print(to_markdown_table(toy_tasks, ["id", "category", "verifier", "answer"]))

## Step 5: Make reward design concrete

Good reward design usually decomposes into small pieces:

- **task reward**: did the answer verify?
- **format reward**: did the output follow the expected contract?
- **process reward**: did the response include the useful structure you asked for?
- **length penalty**: did the model become pointlessly long?

That is more robust than one giant opaque score.

In [ ]:
def extract_final_answer(text):
    match = re.search(r"FINAL:\s*(.+)", text.strip(), flags=re.IGNORECASE)
    return match.group(1).strip() if match else None

def format_reward(text):
    return 0.2 if extract_final_answer(text) is not None else -0.2

def reasoning_reward(text):
    lowered = text.lower()
    return 0.15 if ("because" in lowered or "step" in lowered or "compute" in lowered) else 0.0

def length_penalty(text, target_words=18):
    words = text.split()
    overflow = max(0, len(words) - target_words)
    return -0.01 * overflow

def verifier_reward(task, text):
    final_answer = extract_final_answer(text)
    return 1.0 if final_answer == task["answer"] else 0.0

def total_reward(task, text):
    pieces = {
        "verifier": verifier_reward(task, text),
        "format": format_reward(text),
        "reasoning": reasoning_reward(text),
        "length": round(length_penalty(text), 3),
    }
    pieces["total"] = round(sum(pieces.values()), 3)
    return pieces

reward_examples = [
    {
        "task_id": "sum_12_19",
        "output": "Compute 12 + 19 = 31. FINAL: 31",
    },
    {
        "task_id": "sum_12_19",
        "output": "31",
    },
    {
        "task_id": "sum_12_19",
        "output": "Because 12 + 19 = 31, the result is clear. FINAL: 31",
    },
    {
        "task_id": "parity_41",
        "output": "41 divides by 2 cleanly, so it is even. FINAL: even",
    },
]

task_lookup = {task["id"]: task for task in toy_tasks}
reward_rows = []
for example in reward_examples:
    pieces = total_reward(task_lookup[example["task_id"]], example["output"])
    reward_rows.append({
        "task_id": example["task_id"],
        "output": example["output"],
        **pieces,
    })

print(to_markdown_table(reward_rows, ["task_id", "verifier", "format", "reasoning", "length", "total", "output"]))

## Step 6: See how GRPO uses group-relative rewards

GRPO does not just ask whether a response is good in absolute terms. It compares sampled rollouts **within the same prompt group** and reinforces responses that beat the group average.

In [ ]:
rollout_group = [
    "Compute 12 + 19 = 31. FINAL: 31",
    "Because 12 + 19 = 31, the answer is 31. FINAL: 31",
    "I think it is 30. FINAL: 30",
    "31",
]

task = task_lookup["sum_12_19"]
raw_rewards = [total_reward(task, text)["total"] for text in rollout_group]

def group_relative_scores(values):
    mean_value = statistics.mean(values)
    stdev = statistics.pstdev(values)
    if stdev == 0:
        return [0.0 for _ in values]
    return [round((value - mean_value) / stdev, 3) for value in values]

advantages = group_relative_scores(raw_rewards)
group_rows = []
for text, reward, advantage in zip(rollout_group, raw_rewards, advantages):
    group_rows.append({
        "reward": reward,
        "advantage": advantage,
        "output": text,
    })

print(to_markdown_table(group_rows, ["reward", "advantage", "output"]))

## Step 7: Distinguish RLHF, RLVR, and GRPO

These names are related but not interchangeable:

- **RLHF** usually learns a reward model from human comparisons and then optimizes against that learned reward.
- **RLVR** uses a direct verifier instead of a learned reward model.
- **GRPO** is a training procedure that uses group-relative normalization and is often paired with verifier rewards on reasoning tasks.

So a common 2026 pattern is: **GRPO + verifier rewards = a practical RLVR-style reasoning recipe**.

In [ ]:
comparison_rows = [
    {
        "approach": "RLHF",
        "reward_source": "learned reward model",
        "human_labels_needed": "many",
        "best_fit": "subjective helpfulness or style",
        "main_failure": "reward model misgeneralizes",
    },
    {
        "approach": "RLVR",
        "reward_source": "programmatic verifier",
        "human_labels_needed": "few",
        "best_fit": "math, code, schema, tool correctness",
        "main_failure": "reward too narrow",
    },
    {
        "approach": "GRPO",
        "reward_source": "relative reward inside sampled group",
        "human_labels_needed": "depends on reward",
        "best_fit": "reasoning rollouts with multiple samples",
        "main_failure": "group gaming and rollout cost",
    },
]

print(to_markdown_table(comparison_rows, ["approach", "reward_source", "human_labels_needed", "best_fit", "main_failure"]))

## Step 8: Design simple verifiers, not vague judges

A good verifier is:

- deterministic
- cheap to run
- aligned with the real success condition
- hard to game accidentally

Below we implement a tiny arithmetic verifier and inspect edge cases.

In [ ]:
def arithmetic_verifier(task, text):
    parsed = extract_final_answer(text)
    result = {
        "task_id": task["id"],
        "parsed_answer": parsed,
        "correct": parsed == task["answer"],
        "has_final_tag": parsed is not None,
        "word_count": len(text.split()),
    }
    return result

edge_cases = [
    "FINAL: 31",
    "The result is 31 and FINAL: 31",
    "FINAL: thirty one",
    "31 FINAL: 31 maybe 32",
    "No final tag here",
]

verifier_rows = [arithmetic_verifier(task_lookup["sum_12_19"], text) for text in edge_cases]
print(to_markdown_table(verifier_rows, ["task_id", "parsed_answer", "correct", "has_final_tag", "word_count"]))

## Step 9: Inspect reward hacking and failure modes

Rewards can be gamed. Common failures include:

- always emitting the required format but wrong answers
- writing very long rationales because long outputs correlate with reward
- learning shortcuts that exploit parser weaknesses

The next cell makes those failure modes visible.

In [ ]:
reward_hacking_cases = [
    {
        "label": "format_only",
        "text": "FINAL: 31",
    },
    {
        "label": "verbose_correct",
        "text": "Because we can decompose 12 + 19 into 12 + 18 + 1, we reach 31 after a small adjustment. FINAL: 31",
    },
    {
        "label": "wrong_but_formatted",
        "text": "Because 12 + 19 feels close to 30, I will answer that. FINAL: 30",
    },
    {
        "label": "parser_attack",
        "text": "FINAL: 31\nFINAL: 999",
    },
]

def verifier_diagnostics(task, text):
    pieces = total_reward(task, text)
    parsed = extract_final_answer(text)
    return {
        "parsed": parsed,
        "reward": pieces["total"],
        "verifier": pieces["verifier"],
        "format": pieces["format"],
        "reasoning": pieces["reasoning"],
        "length": pieces["length"],
    }

diagnostic_rows = []
for case in reward_hacking_cases:
    diagnostic_rows.append({
        "label": case["label"],
        **verifier_diagnostics(task_lookup["sum_12_19"], case["text"]),
        "text": case["text"],
    })

print(to_markdown_table(diagnostic_rows, ["label", "parsed", "verifier", "format", "reasoning", "length", "reward", "text"]))

## Step 10: Why RL is not the first lever

If prompt fixes, retrieval, better demonstrations, or preference tuning already solve the problem, RL adds cost without enough benefit. The next cell forces that decision into the open.

In [ ]:
interventions = [
    {"lever": "prompting", "setup_cost": 0.1, "maint_cost": 0.1, "signal_need": 0.1},
    {"lever": "better data", "setup_cost": 0.4, "maint_cost": 0.3, "signal_need": 0.2},
    {"lever": "preference tuning", "setup_cost": 0.55, "maint_cost": 0.45, "signal_need": 0.45},
    {"lever": "GRPO", "setup_cost": 0.85, "maint_cost": 0.8, "signal_need": 0.9},
]

product_needs = {
    "exact_verification": 0.95,
    "subjective_style": 0.3,
    "current_eval_strength": 0.8,
}

def choose_first_lever(needs):
    rows = []
    for item in interventions:
        leverage = (
            0.55 * needs["exact_verification"] * item["signal_need"]
            + 0.25 * needs["current_eval_strength"]
            - 0.20 * item["setup_cost"]
            - 0.10 * item["maint_cost"]
        )
        rows.append({
            "lever": item["lever"],
            "score": round(leverage, 3),
            "score_bar": ascii_bar(max(0.0, min(1.0, leverage))),
        })
    return sorted(rows, key=lambda row: row["score"], reverse=True)

priority_rows = choose_first_lever(product_needs)
print(to_markdown_table(priority_rows, ["lever", "score", "score_bar"]))

## Step 11: Turn the notebook into GRPO-ready artifacts

Even a foundations notebook can produce useful assets: prompts, answers, verifier names, and reward components you can reuse later.

In [ ]:
prompt_reward_records = []
for task in toy_tasks:
    prompt_reward_records.append({
        "prompt": task["prompt"],
        "expected_answer": task["answer"],
        "verifier": task["verifier"],
        "reward_components": ["verifier", "format", "reasoning", "length"],
    })

jsonl_path = ARTIFACT_DIR / "grpo_foundation_tasks.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as handle:
    for record in prompt_reward_records:
        handle.write(json.dumps(record) + "\n")

summary_path = ARTIFACT_DIR / "rl_readiness_cases.json"
with open(summary_path, "w", encoding="utf-8") as handle:
    json.dump(decision_cases, handle, indent=2)

print("Saved:", jsonl_path)
print("Saved:", summary_path)
print("Records:", len(prompt_reward_records))

## Recap

You now have a first-principles view of GRPO:

- RL is justified when the task is verifiable and the reward is trustworthy
- RLHF, RLVR, and GRPO solve different supervision problems
- reward design should be decomposed and inspectable
- verifiers should be deterministic and hard to game
- RL is powerful, but it is not the first lever for most product problems